# RAG From Scratch — NVIDIA NIM API
### Author: Dr Manish Kumar Jain

**What this notebook does:**
1. Loads a PDF document
2. Splits text into chunks
3. Generates embeddings via NVIDIA embedding model
4. Stores vectors in FAISS
5. Retrieves relevant chunks for a user query
6. Sends context + query to NVIDIA LLM for a grounded answer

**Required packages:**
```
pip install openai sentence-transformers faiss-cpu pypdf
```

## Section 1 — Install and Import Dependencies

In [1]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================
# pip install openai sentence-transformers faiss-cpu pypdf

import os
import faiss
import numpy as np
from openai import OpenAI
from pypdf import PdfReader

print("All libraries imported successfully.")

All libraries imported successfully.


## Section 2 — Configure NVIDIA API

Get your free API key from [https://build.nvidia.com](https://build.nvidia.com)

In [2]:
# ============================================================
# CONFIGURE NVIDIA API
# ============================================================

NVIDIA_API_KEY = "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"
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

# LLM Model for answer generation
LLM_MODEL = "meta/llama-3.1-8b-instruct"

# Embedding Model for vector search
EMBEDDING_MODEL = "nvidia/nv-embedqa-e5-v5"

# ============================================================
# INITIALIZE NVIDIA CLIENT
# ============================================================

client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY
)

print(f"NVIDIA client initialized.")
print(f"LLM Model     : {LLM_MODEL}")
print(f"Embedding Model: {EMBEDDING_MODEL}")

NVIDIA client initialized.
LLM Model     : meta/llama-3.1-8b-instruct
Embedding Model: nvidia/nv-embedqa-e5-v5


## Section 3 — Load and Preprocess PDF Documents

In [3]:
# ============================================================
# LOAD PDF FILE
# ============================================================

def load_pdf(pdf_path):
    """Extract all text from a PDF file."""
    text = ""
    reader = PdfReader(pdf_path)

    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"

    print(f"PDF loaded. Total characters: {len(text)}")
    return text

print("load_pdf() function defined.")

load_pdf() function defined.


## Section 4 — Split Text into Chunks

In [4]:
# ============================================================
# SPLIT TEXT INTO OVERLAPPING CHUNKS
# ============================================================
#
# chunk_size  : number of characters per chunk
# overlap     : characters shared between consecutive chunks
#               helps preserve context at chunk boundaries
#

def split_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping fixed-size chunks."""
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap   # step back by overlap

    print(f"Text split into {len(chunks)} chunks.")
    return chunks

print("split_text() function defined.")

split_text() function defined.


## Section 5 — Generate Embeddings with NVIDIA Embedding Model

In [6]:
# ============================================================
# GENERATE EMBEDDINGS USING NVIDIA EMBEDDING MODEL
# ============================================================
#
# Model: nvidia/nv-embedqa-e5-v5
# - High quality dense embeddings
# - Optimised for retrieval tasks
#
# input_type="passage" : used when encoding documents
# input_type="query"   : used when encoding the user question
#

def create_embeddings(texts, input_type="passage"):
    """Generate float32 embeddings for a list of texts via NVIDIA API."""
    response = client.embeddings.create(
        input=texts,
        model=EMBEDDING_MODEL,
        encoding_format="float",
        extra_body={"input_type": input_type, "truncate": "END"}
    )
    embeddings = [item.embedding for item in response.data]
    return np.array(embeddings, dtype="float32")

print("create_embeddings() function defined.")

create_embeddings() function defined.


## Section 6 — Build FAISS Vector Index

In [7]:
# ============================================================
# BUILD FAISS INDEX
# ============================================================
#
# IndexFlatL2 : exact L2 (Euclidean) nearest-neighbour search
# dimension   : must match the embedding size of the model
#

def build_faiss_index(embeddings):
    """Create and populate a FAISS L2 index from embeddings."""
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    print(f"FAISS index built. Vectors stored: {index.ntotal}  Dimension: {dimension}")
    return index

print("build_faiss_index() function defined.")

build_faiss_index() function defined.


## Section 7 — Vector Search Retrieval

In [8]:
# ============================================================
# RETRIEVE RELEVANT CHUNKS
# ============================================================
#
# Steps:
# 1. Encode the user query with input_type="query"
# 2. Search FAISS for top_k nearest vectors
# 3. Return corresponding text chunks
#

def retrieve_chunks(query, index, chunks, top_k=3):
    """Find the top_k most relevant chunks for a user query."""
    query_embedding = create_embeddings([query], input_type="query")
    distances, indices = index.search(query_embedding, top_k)

    retrieved = [chunks[idx] for idx in indices[0] if idx < len(chunks)]
    print(f"Retrieved {len(retrieved)} chunks for query: '{query}'")
    return retrieved

print("retrieve_chunks() function defined.")

retrieve_chunks() function defined.


## Section 8 — Generate Answers with NVIDIA LLM

In [9]:
# ============================================================
# GENERATE ANSWER USING NVIDIA LLM
# ============================================================
#
# Model: meta/llama-3.1-8b-instruct
#
# The prompt is structured as:
#   - System instruction (grounding the model to context only)
#   - Retrieved context chunks
#   - User question
#

def ask_nvidia(query, retrieved_chunks):
    """Send retrieved context + query to NVIDIA LLM and return the answer."""
    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f"""You are a helpful AI assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't know based on the provided document."

Context:
{context}

Question:
{query}

Answer:"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=512
    )

    return response.choices[0].message.content

print("ask_nvidia() function defined.")

ask_nvidia() function defined.


## Section 9 — Interactive RAG Query Loop

Place your PDF file in `c:\Aristocrat\AITraining\` and update `pdf_path` below.

In [10]:
# ============================================================
# MAIN RAG PIPELINE
# ============================================================

# ---- SET YOUR PDF PATH HERE --------------------------------
pdf_path = "sample.pdf"
# -----------------------------------------------------------

# Step 1 : Load PDF
print("=" * 50)
print("Step 1 : Loading PDF...")
document_text = load_pdf(pdf_path)

# Step 2 : Split into chunks
print("\nStep 2 : Splitting text into chunks...")
chunks = split_text(document_text, chunk_size=500, overlap=50)

# Step 3 : Generate embeddings for all chunks
print("\nStep 3 : Generating embeddings via NVIDIA API...")
embeddings = create_embeddings(chunks, input_type="passage")
print(f"Embeddings shape: {embeddings.shape}")

# Step 4 : Build FAISS index
print("\nStep 4 : Building FAISS vector index...")
index = build_faiss_index(embeddings)

print("\n" + "=" * 50)
print("RAG SYSTEM READY")
print("Type your question below. Type 'exit' to quit.")
print("=" * 50)

# Step 5 : Interactive query loop
while True:
    query = input("\nEnter Question: ").strip()

    if not query:
        continue

    if query.lower() == "exit":
        print("Goodbye!")
        break

    # Retrieve relevant chunks
    retrieved = retrieve_chunks(query, index, chunks, top_k=3)

    # Generate answer from NVIDIA LLM
    print("\nGenerating answer from NVIDIA LLM...")
    answer = ask_nvidia(query, retrieved)

    print("\n" + "=" * 50)
    print("ANSWER:")
    print(answer)
    print("=" * 50)

Step 1 : Loading PDF...
PDF loaded. Total characters: 15238

Step 2 : Splitting text into chunks...
Text split into 34 chunks.

Step 3 : Generating embeddings via NVIDIA API...
Embeddings shape: (34, 1024)

Step 4 : Building FAISS vector index...
FAISS index built. Vectors stored: 34  Dimension: 1024

RAG SYSTEM READY
Type your question below. Type 'exit' to quit.
Retrieved 3 chunks for query: 'what are the models'

Generating answer from NVIDIA LLM...

ANSWER:
Based on the provided context, the models mentioned are:

1. T5
2. FLAN
3. GPT
Goodbye!
